# GLiNER2 한국어 추출: 언어와 체크포인트

GLiNER2.5의 두 공개 체크포인트가 한국어 문장에서 엔티티와 관계를 얼마나 정확히 추출하는지, 같은 내용의 영어 문장과 LLM 추출을 기준으로 비교한다.

| 체크포인트 | 인코더 | 비고 |
|---|---|---|
| [`fastino/gliner2.5-base-v1`](https://huggingface.co/fastino/gliner2.5-base-v1) | 영어 중심 | `neo4j-agent-memory`의 기본 체크포인트 |
| [`fastino/gliner2.5-multi-v1`](https://huggingface.co/fastino/gliner2.5-multi-v1) | mDeBERTa-v3-base (다국어) | 다국어 용도 권장 체크포인트 |

입력은 실제 대화에서 가져온 한국어 문장 4개와 이를 직접 번역한 영어 문장 4개이다. 같은 의미의 문장 쌍을 비교하여 오류가 언어에서 오는지, 표기(한글/로마자)나 띄어쓰기에서 오는지를 분리한다.

> 이 실험은 [`neo4j/agent-memory`](../../../neo4j/agent-memory/)에서 SDK 경유로 관찰한 GLiNER 오류를 SDK 없이 재현·분석하기 위해 시작했다. SDK 통합 관점의 결과는 해당 프로젝트의 [Hybrid와 LLM-only 비교](../../../neo4j/agent-memory/experiments/01_entity_extraction_hybrid_vs_llm.ipynb)에 있다.

## 결과 요약

> 아래 수치는 2026-11-10 실행 결과(`language_and_checkpoint.json`, 문장 쌍 4개 × 언어 2개 × 3회)이다.

| 추출기 | 한국어 문장 1 (GCP) | 한국어 문장 4 (오라클) | 영어 | 중앙값 지연 (ko / en) |
|---|---|---|---|---|
| `gliner_base` | `GCP에서` ✗, Cloud DNS 누락 | **PERSON ✗** | Oracle ORG ✓, 문장 1 엔티티 없음 | 119 / 128 ms |
| `gliner_multi` | `GCP에서` ✗, Cloud DNS 누락 | ORGANIZATION ✓ | `I(PERSON)`, `November(LOCATION)` 등 노이즈 | 136 / 140 ms |
| `llm_only` | `GCP` ✓, `Cloud DNS` ✓ | ORGANIZATION ✓ | 정상 | 1,888 / 1,854 ms |

- **조사 문제는 두 체크포인트 공통이다.** `GCP 에서`로 띄어 쓰면 둘 다 `GCP(ORGANIZATION)`을 정확히 추출한다. 원인은 모델이 아니라 공백 단위 단어 분할이며, [단어 분할기 비교](02_korean_word_splitter.ipynb)에서 해결 방법을 확인한다.
- **오라클 타입 오류는 표기와 문맥의 조합에 따라 달라진다.** base는 한국어 문맥의 `오라클`을, multi는 영어 문맥의 `오라클`을 PERSON으로 분류한다.
- **한국어에서 GLiNER는 관계를 하나도 생성하지 못했다.** 영어에서도 multi의 관계는 `I -PARTICIPATED_IN→ usage` 같은 무의미한 결과뿐이다.
- GLiNER는 LLM보다 약 14배 빠르지만, 이 조건에서는 품질 차이가 속도 이점보다 크다.

## 실험 조건

| 항목 | 값 |
|---|---|
| 추출기 | GLiNER base, GLiNER multi (CPU, 공백 분할기), LLM |
| 스키마 | POLE+O 엔티티 5종, 관계 15종 |
| 반복 | 문장 쌍 4개 × 언어 2개 × 3회, 반복마다 언어·추출기 순서를 교대 |
| 지연 시간 | 워밍업 이후 측정. LLM은 네트워크 왕복 포함 |

| # | 한국어 | 영어 (직접 번역) |
|---|---|---|
| 1 | 이제 GCP에서 매달 나가는 비용은 Cloud DNS 영역 3개, 월 약 $0.6뿐입니다. | The only recurring GCP cost now is for three Cloud DNS zones, about $0.60 per month. |
| 2 | 화면과 상관없는 서버 쪽 작업(공지 저장, 작업실 정보)은 이미 같이 진행하고 있어요. | The server-side tasks unrelated to the screens (saving announcements and studio information) are already being worked on in parallel. |
| 3 | 11월 사용량을 서비스별로 조회해 보니 금액이 붙은 항목이 하나도 없습니다. | I checked November usage by service, and none of the items had any charges. |
| 4 | 오라클 공식 입장: 고정 IP 기능을 처음 발표할 때 'There is no charge to use this feature'라고 밝혔습니다. | Oracle's official position: when it first announced the static IP feature, it stated, 'There is no charge to use this feature'. |

## 준비

### 경고 출력 설정

체크포인트 로딩 시 출력되는 Transformers·PyTorch 경고를 숨긴다. 경고의 원인을 처리하지는 않는다.

In [ ]:
import logging
import warnings

warnings.filterwarnings("ignore", category=UserWarning)
logging.getLogger("transformers").setLevel(logging.ERROR)

### 프로젝트 경로와 환경 변수

현재 작업 디렉터리에서 상위로 올라가며 `pyproject.toml`이 있는 프로젝트 루트를 찾아 모듈 검색 경로에 추가한다. 추출기와 스키마는 프로젝트의 `config` 모듈에서 가져온다.
`.env`에서 `OPENAI_API_KEY`와 LLM 모델 이름(`LLM_MODEL`)을 읽는다. LLM 기준선에만 필요하다.

In [ ]:
import os
from pathlib import Path
import sys

from dotenv import load_dotenv

PROJECT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").exists())
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
load_dotenv(PROJECT / ".env")
LLM_MODEL = os.getenv("LLM_MODEL", "gpt-5.4-mini")
RESULT_PATH = PROJECT / "experiments/results/language_and_checkpoint.json"
print(f"✓ Project root: {PROJECT}")
print(f"✓ Result file: {RESULT_PATH.relative_to(PROJECT)}  LLM: {LLM_MODEL}")

### 실행 방식 선택

`RUN_EXPERIMENT = False`이면 저장된 결과 파일을 불러와 분석만 수행한다.
`True`로 바꾸면 GLiNER 체크포인트를 내려받아 로딩하고 LLM API를 호출하여 결과 파일을 덮어쓴다.

In [ ]:
RUN_EXPERIMENT = False
REPEATS = 3

### 보조 함수

결과 파일 저장·불러오기와 분석 표 출력에 쓰는 함수이다.

- `entity_summary()`: 엔티티별로 `이름(타입) 등장 횟수/실행 수 · 최대 confidence`를 이어 붙인다.
- `relation_summary()`: 관계별로 `출발점 -타입→ 도착점 등장 횟수/실행 수`를 이어 붙인다.
- `has_particle()`: Kiwi로 분석한 엔티티 이름의 마지막 형태소가 조사(`J*`)인지 판정한다.

In [ ]:
from collections import Counter
import json

from IPython.display import Markdown, display

from config.extractors import kiwi


def save(path, report):
    path.write_text(json.dumps(report, ensure_ascii=False, indent=2) + "\n")


def load(path):
    return json.loads(path.read_text())


def show_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "|" + "---|" * len(headers)]
    lines += ["| " + " | ".join(str(cell) for cell in row) + " |" for row in rows]
    display(Markdown("\n".join(lines)))


def entity_summary(runs):
    counts, confidence = Counter(), {}
    for run in runs:
        for entity in run["result"]["entities"]:
            key = (entity["name"], entity["type"])
            confidence[key] = max(confidence.get(key, 0), entity["confidence"])
        counts.update({(e["name"], e["type"]) for e in run["result"]["entities"]})
    items = sorted(counts.items(), key=lambda item: (-item[1], item[0]))
    return "<br>".join(f"{n}({t}) {c}/{len(runs)} · {confidence[n, t]:.2f}"
                       for (n, t), c in items) or "–"


def relation_summary(runs):
    counts = Counter()
    for run in runs:
        counts.update({(r["source"], r["type"], r["target"]) for r in run["result"]["relations"]})
    return "<br>".join(f"{s} -{t}→ {o} {c}/{len(runs)}"
                       for (s, t, o), c in sorted(counts.items(), key=lambda i: (-i[1], i[0]))) or "–"


def has_particle(name):
    tokens = kiwi().tokenize(name)
    return bool(tokens) and tokens[-1].tag.startswith("J")

### 입력 문장과 통제 프로브

통제 프로브는 문장 하나에서 한 요소만 바꾼 입력이다. 원인 진단용이며 운영 전처리 방안이 아니다.

| 프로브 | 변경 | 확인하려는 것 |
|---|---|---|
| `ko_context_latin_oracle` | 한국어 문장 4의 `오라클` → `Oracle` | 한국어 문맥에서 표기만 바꾸면 타입이 바뀌는가 |
| `en_context_hangul_oracle` | 영어 문장 4의 `Oracle` → `오라클` | 영어 문맥에서 한글 표기를 쓰면 타입이 바뀌는가 |
| `ko_gcp_particle_separated` | 한국어 문장 1의 `GCP에서` → `GCP 에서` | 조사를 띄어 쓰면 엔티티 범위가 정확해지는가 |

In [ ]:
KOREAN = [
    "이제 GCP에서 매달 나가는 비용은 Cloud DNS 영역 3개, 월 약 $0.6뿐입니다.",
    "화면과 상관없는 서버 쪽 작업(공지 저장, 작업실 정보)은 이미 같이 진행하고 있어요.",
    "11월 사용량을 서비스별로 조회해 보니 금액이 붙은 항목이 하나도 없습니다.",
    "오라클 공식 입장: 고정 IP 기능을 처음 발표할 때 'There is no charge to use this feature'라고 밝혔습니다.",
]
ENGLISH = [
    "The only recurring GCP cost now is for three Cloud DNS zones, about $0.60 per month.",
    "The server-side tasks unrelated to the screens (saving announcements and studio information) are already being worked on in parallel.",
    "I checked November usage by service, and none of the items had any charges.",
    "Oracle's official position: when it first announced the static IP feature, it stated, 'There is no charge to use this feature'.",
]
TEXTS = {"ko": KOREAN, "en": ENGLISH}
PROBES = {
    "ko_context_latin_oracle": KOREAN[3].replace("오라클", "Oracle"),
    "en_context_hangul_oracle": ENGLISH[3].replace("Oracle", "오라클"),
    "ko_gcp_particle_separated": KOREAN[0].replace("GCP에서", "GCP 에서"),
}
CHECKPOINTS = {"gliner_base": "fastino/gliner2.5-base-v1", "gliner_multi": "fastino/gliner2.5-multi-v1"}

### 추출기

[`config/extractors.py`](../config/extractors.py)의 두 추출기는 같은 POLE+O 스키마([`config/ontology.py`](../config/ontology.py))를 사용하고 같은 형태의 결과를 반환한다.

| 추출기 | 동작 |
|---|---|
| `GlinerExtractor` | `gliner2`의 JointIE 엔진에 엔티티 타입과 관계 타입(머리·꼬리 타입, 설명, threshold)을 스키마로 등록하고, 엔티티와 관계를 한 번에 디코딩한다. 엔티티 threshold는 0.2이다. |
| `LLMExtractor` | 같은 타입 정의를 지시문으로 전달하고 OpenAI 구조화 출력(JSON Schema)으로 엔티티와 관계를 받는다. 존재하지 않는 엔티티를 가리키거나 온톨로지가 허용하지 않는 타입 조합의 관계는 버린다. |

## 실험

### 반복 추출과 프로브

1. 모든 추출기를 짧은 영어 문장으로 한 번씩 워밍업하여 모델 로딩 시간을 측정에서 제외한다.
2. 문장 쌍마다 언어별로 세 추출기를 실행하고 결과와 실행 시간을 기록한다.
3. GLiNER 두 체크포인트에 통제 프로브를 입력한다.

In [ ]:
from datetime import datetime, timezone
from importlib.metadata import version
import platform
from time import perf_counter

from config.extractors import GlinerExtractor, LLMExtractor


async def run_experiment(repeats):
    extractors = {mode: GlinerExtractor(model) for mode, model in CHECKPOINTS.items()}
    llm = LLMExtractor(LLM_MODEL)

    async def extract(mode, text):
        return await llm.extract(text) if mode == "llm_only" else extractors[mode].extract(text)

    modes = [*extractors, "llm_only"]
    report = {
        "status": "running", "started_at": datetime.now(timezone.utc).isoformat(),
        "repeats": repeats, "llm_model": LLM_MODEL, "checkpoints": CHECKPOINTS,
        "versions": {p: version(p) for p in ("gliner2", "torch", "openai")},
        "hardware": {"system": platform.system(), "architecture": platform.machine(), "device": "cpu"},
        "texts": TEXTS, "warmup": {}, "runs": [], "probes": [],
    }
    for mode in modes:
        started = perf_counter()
        await extract(mode, "Alice works at Acme in Seoul.")
        report["warmup"][mode] = round((perf_counter() - started) * 1000, 1)

    for repeat in range(1, repeats + 1):
        languages = ("ko", "en") if repeat % 2 else ("en", "ko")
        order = modes if repeat % 2 else modes[::-1]
        for index in range(1, len(KOREAN) + 1):
            for language in languages:
                for mode in order:
                    started = perf_counter()
                    result = await extract(mode, TEXTS[language][index - 1])
                    report["runs"].append({
                        "mode": mode, "language": language, "repeat": repeat, "message_index": index,
                        "elapsed_ms": round((perf_counter() - started) * 1000, 1), "result": result,
                    })
        save(RESULT_PATH, report)
        print(f"✓ repeat {repeat}/{repeats}")

    for probe, text in PROBES.items():
        for mode in extractors:
            report["probes"].append({"probe": probe, "mode": mode, "text": text,
                                     "result": extractors[mode].extract(text)})
    report["status"] = "complete"
    report["completed_at"] = datetime.now(timezone.utc).isoformat()
    save(RESULT_PATH, report)
    return report

### 실행 또는 결과 불러오기

In [ ]:
report = await run_experiment(REPEATS) if RUN_EXPERIMENT else load(RESULT_PATH)
MODES = list(dict.fromkeys(run["mode"] for run in report["runs"]))
print(f"✓ status={report['status']} runs={len(report['runs'])} probes={len(report['probes'])}")
print(f"  versions={report['versions']} llm={report['llm_model']}")

## 결과 분석

표는 `report`에서 다시 계산한다. 셀 출력 없이 읽을 수 있도록, 각 표 아래에 실행 결과의 값과 해석을 함께 기록한다.

In [ ]:
from statistics import median


def runs_of(mode, language=None, index=None):
    return [r for r in report["runs"] if r["mode"] == mode
            and (language is None or r["language"] == language)
            and (index is None or r["message_index"] == index)]


def count(runs, predicate):
    """Number of runs in which at least one entity satisfies the predicate."""
    return f"{sum(any(predicate(e) for e in r['result']['entities']) for r in runs)}/{len(runs)}"

### 관찰 지표

문장 1(GCP)과 문장 4(오라클)에서 문제가 된 엔티티가 3회 중 몇 번의 실행에서 나타났는지 센다. `조사 포함`은 한국어 문장 전체에서 이름의 마지막 형태소가 조사인 엔티티의 실행당 평균 개수이다.

In [ ]:
for language, oracle in (("ko", "오라클"), ("en", "Oracle")):
    display(Markdown(f"**{'한국어' if language == 'ko' else '영어'}**"))
    rows = []
    for mode in MODES:
        first, fourth, every = (runs_of(mode, language, 1), runs_of(mode, language, 4),
                                runs_of(mode, language))
        particles = sum(has_particle(e["name"]) for r in every for e in r["result"]["entities"])
        rows.append([
            mode,
            count(first, lambda e: e["name"] == "GCP" and e["type"] == "ORGANIZATION"),
            count(first, lambda e: e["name"] == "GCP에서"),
            count(first, lambda e: e["name"].startswith("Cloud DNS")),
            count(fourth, lambda e: e["name"] == oracle and e["type"] == "ORGANIZATION"),
            count(fourth, lambda e: e["name"] == oracle and e["type"] == "PERSON"),
            f"{particles / report['repeats']:.1f}" if language == "ko" else "–",
            f'{sum(len(r["result"]["relations"]) for r in every) / report["repeats"]:.1f}',
            f"{median(r['elapsed_ms'] for r in every):,.0f}",
        ])
    show_table(["모드", "GCP(ORG)", "GCP에서", "Cloud DNS*", f"{oracle}(ORG)", f"{oracle}(PERSON)",
                "조사 포함", "관계 수/회", "중앙값 ms"], rows)

**한국어**

| 모드 | GCP(ORG) | GCP에서 | Cloud DNS | 오라클(ORG) | 오라클(PERSON) | 조사 포함 | 관계 수/회 | 중앙값 ms |
|---|---|---|---|---|---|---|---|---|
| gliner_base | 0/3 | **3/3** | 0/3 | 0/3 | **3/3** | 1.0 | 0.0 | 119 |
| gliner_multi | 0/3 | **3/3** | 0/3 | 3/3 | 0/3 | 1.0 | 0.0 | 136 |
| llm_only | 3/3 | 0/3 | 3/3 | 3/3 | 0/3 | 0.0 | 3.3 | 1,888 |

**영어**

| 모드 | GCP(ORG) | Cloud DNS* | Oracle(ORG) | Oracle(PERSON) | 관계 수/회 | 중앙값 ms |
|---|---|---|---|---|---|---|
| gliner_base | 0/3 | 0/3 | 3/3 | 0/3 | 0.0 | 128 |
| gliner_multi | 3/3 | 0/3 | 3/3 | 0/3 | 2.0 | 140 |
| llm_only | 3/3 | 3/3 | 3/3 | 0/3 | 1.3 | 1,854 |

\* `Cloud DNS`로 시작하는 이름(예: `Cloud DNS zones`)을 인정한다.

**해석**

- 한국어에서 GLiNER는 두 체크포인트 모두 `GCP`를 조사와 함께 추출하고 `Cloud DNS`를 찾지 못한다. 영어에서는 multi가 `GCP`를 정확히 추출하므로, 한국어 입력에서 품질이 크게 떨어진다.
- base의 오라클 PERSON 오류는 한국어에서만 나타난다. multi는 한국어 문장 4를 정확히 분류한다.

### 문장별 엔티티

`n/3`은 3회 중 등장 횟수, 마지막 숫자는 최대 confidence이다.

In [ ]:
for index in range(1, len(KOREAN) + 1):
    display(Markdown(f"**문장 {index}** — {KOREAN[index - 1]}"))
    show_table(["모드", "한국어", "영어"],
               [[mode, entity_summary(runs_of(mode, "ko", index)), entity_summary(runs_of(mode, "en", index))]
                for mode in MODES])

**해석**

- GLiNER는 같은 입력에 3회 모두 같은 결과를 반환한다. LLM은 회차마다 결과가 달라진다(예: 한국어 문장 1의 `Cloud DNS` 타입이 OBJECT 2회, ORGANIZATION 1회).
- multi는 한국어 문장 2·3에서 `공지 저장(EVENT)`, `11월(EVENT)`을 추출하지만, 영어 문장 3에서는 `I(PERSON)`, `November(LOCATION) 0.09` 같은 명백한 오류를 낸다.
- LLM도 노이즈가 있다. 한국어에서 `월 약 $0.6(OBJECT)`, `서비스별(OBJECT)`, 인용문 전체(`There is no charge to use this feature`)를 엔티티로 만든다.
- 영어 문장 3에서 GLiNER 두 체크포인트는 `service`를 ORGANIZATION으로 0.97~0.99의 높은 confidence로 분류한다. 오류의 confidence가 정답과 구분되지 않는다.

### 관계

In [ ]:
show_table(["#", *(f"{mode} ({lang})" for mode in MODES for lang in ("ko", "en"))],
           [[index, *(relation_summary(runs_of(mode, lang, index)) for mode in MODES for lang in ("ko", "en"))]
            for index in range(1, len(KOREAN) + 1)])

**해석**

- 한국어에서 GLiNER의 관계는 0개이다. GLiNER는 자신이 찾은 엔티티 사이에서만 관계를 만들며, 엔티티를 1~2개만 찾거나 조사가 붙은 범위로 찾은 문장에서는 관계 후보 자체가 없다.
- 영어 multi의 관계 2개(`I -PARTICIPATED_IN→ usage`, `usage -OCCURRED_AT→ November`)는 잘못 추출한 엔티티 위에 만든 관계라 모두 틀렸다.
- LLM의 관계는 대부분 `RELATED_TO`이다. 이 4개 문장에는 구체적인 관계가 거의 없어 관계 품질 비교는 [단어 분할기 비교](02_korean_word_splitter.ipynb)의 문장 5~9에서 수행한다.

### 통제 프로브

In [ ]:
probes = {}
for probe in report["probes"]:
    probes.setdefault(probe["probe"], {"text": probe["text"]})[probe["mode"]] = ", ".join(
        f"{e['name']}({e['type']})" for e in probe["result"]["entities"]) or "–"
show_table(["프로브", "입력", *CHECKPOINTS],
           [[label, item["text"], *(item[mode] for mode in CHECKPOINTS)] for label, item in probes.items()])

| 프로브 | gliner_base | gliner_multi |
|---|---|---|
| 한국어 문맥 + `Oracle` | Oracle(ORG) | Oracle(ORG) |
| 영어 문맥 + `오라클` | 오라클(ORG), static IP feature(OBJECT) | **오라클(PERSON)**, static IP feature(EVENT) |
| `GCP 에서` (조사 띄어쓰기) | **GCP(ORG)** | **GCP(ORG)** |

**해석**

- **조사**: 조사를 띄어 쓰면 두 체크포인트 모두 정확한 범위의 `GCP`를 추출한다. GLiNER는 공백 단위 토큰을 엔티티 경계로 사용하므로, 교착어의 조사를 분리하지 못하면 범위가 틀린다.
- **오라클 타입**: base는 한국어 문맥의 한글 `오라클`에서만 PERSON으로 분류하고, 표기나 문맥 중 하나만 바꾸면 ORGANIZATION으로 분류한다. multi는 반대로 영어 문맥의 한글 `오라클`을 PERSON으로 분류한다. 한국어 대화에는 영어 고유명사의 한글 표기와 영문 표기가 섞이므로, 체크포인트 교체만으로는 이 불안정성을 피할 수 없다.

### 지연 시간

In [ ]:
show_table(["모드", "워밍업 ms", "한국어 중앙값 ms", "영어 중앙값 ms"],
           [[mode, f"{report['warmup'][mode]:,.0f}",
             *(f"{median(r['elapsed_ms'] for r in runs_of(mode, lang)):,.0f}" for lang in ("ko", "en"))]
            for mode in MODES])

| 모드 | 워밍업 ms | 한국어 중앙값 ms | 영어 중앙값 ms |
|---|---|---|---|
| gliner_base | 214 | 119 | 128 |
| gliner_multi | 150 | 136 | 140 |
| llm_only | 4,533 | 1,888 | 1,854 |

워밍업 시간에는 체크포인트 로딩이 포함되지 않는다(추출기 생성 시 로딩). GLiNER는 CPU에서 문장당 약 0.12~0.14초로, 네트워크 왕복을 포함한 LLM보다 약 14배 빠르며 두 체크포인트의 속도 차이는 작다.

## 결론과 한계

**결론**

1. 공개 GLiNER2.5 체크포인트를 공백 분할 그대로 쓰면 한국어 엔티티 범위에 조사가 포함되고, 관계는 거의 생성되지 않는다.
2. multi 체크포인트는 base보다 한국어 타입 판단이 낫지만, 조사 문제는 같고 문자 체계가 섞인 입력에서 타입 판단이 불안정하다.
3. 조사 문제는 띄어쓰기 프로브로 해소되므로, 단어 분할 단계의 개선이 우선 과제이다 → [단어 분할기 비교](02_korean_word_splitter.ipynb).

**한계**

- 문장 쌍 4개 × 3회의 진단용 표본이며 정답 레이블이 없다. 일반적인 언어별 성능 비교가 아니다.
- 영어 문장은 직접 번역한 것으로, 번역 과정에서 표현 차이가 생긴다.
- 지연 시간은 단일 머신(Darwin arm64)의 CPU 실행 결과이며, LLM 시간은 네트워크 상태에 따라 달라진다.